# E-Commerce User Churn Predictor
## EDA, Feature Selection & Model Training

This notebook covers:
1. **Data Loading** -- Load products, carts, users from DummyJSON API
2. **Exploratory Data Analysis** -- Understand cart behavior and multi-order patterns
3. **Churn Label Definition** -- 30-day inactivity threshold (from cart dates)
4. **Feature Engineering** -- 10 features: cart behavior + product engagement (no date features)
5. **Feature Selection** -- 4 methods: Filter, RFE, Random Forest, Decision Tree
6. **Consensus Analysis** -- Compare all methods
7. **Model Training** -- Train final Random Forest classifier
8. **Save Model** -- Export model and feature metadata

In [43]:
import json
import os
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import joblib

sys.path.insert(0, '../app')
from features import EcommerceChurnFeatureEngineer

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
import warnings
warnings.filterwarnings('ignore')
print('All imports successful')

All imports successful


## 1. Load Data

In [44]:
data_dir = '../data/raw'

with open(os.path.join(data_dir, 'products.json')) as f:
    products_data = json.load(f)
with open(os.path.join(data_dir, 'carts.json')) as f:
    carts_data = json.load(f)
with open(os.path.join(data_dir, 'users.json')) as f:
    users_data = json.load(f)

products_df = pd.DataFrame(products_data['products'])
carts_df = pd.DataFrame(carts_data['carts'])
users_df = pd.DataFrame(users_data['users'])

print('Products:', len(products_df))
print('Carts:', len(carts_df))
print('Users:', len(users_df))
print('Product categories:', products_df['category'].nunique())
print('Price range:', products_df['price'].min(), '-', products_df['price'].max())

Products: 194
Carts: 548
Users: 208
Product categories: 24
Price range: 0.79 - 36999.99


## 2. Exploratory Data Analysis

In [45]:
carts_df['date_dt'] = pd.to_datetime(carts_df['date'], unit='ms')
carts_df['days_ago'] = (datetime.now() - carts_df['date_dt']).dt.days

# Per-user stats
user_stats = carts_df.groupby('userId').agg(
    n_carts=('id', 'count'),
    total_items=('totalQuantity', 'sum'),
    total_spent=('total', 'sum'),
    last_active=('days_ago', 'min')
).reset_index()

print('=== CART ANALYSIS ===')
print('Total carts:', len(carts_df))
print('Unique users:', carts_df['userId'].nunique())
print('Carts per user: mean=', round(user_stats['n_carts'].mean(), 1), ', max=', user_stats['n_carts'].max())
print('Items per user: mean=', round(user_stats['total_items'].mean(), 1))
print('Spend per user: mean=$', round(user_stats['total_spent'].mean(), 2))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].hist(user_stats['n_carts'], bins=range(1, user_stats['n_carts'].max()+2), edgecolor='black', alpha=0.7)
axes[0,0].set_title('Orders per User Distribution')
axes[0,0].set_xlabel('Number of Orders')
axes[0,0].set_ylabel('Number of Users')

axes[0,1].hist(user_stats['total_items'], bins=20, edgecolor='black', alpha=0.7, color='orange')
axes[0,1].set_title('Total Items per User')
axes[0,1].set_xlabel('Total Items')

axes[1,0].hist(user_stats['total_spent'], bins=20, edgecolor='black', alpha=0.7, color='green')
axes[1,0].set_title('Total Spend per User')
axes[1,0].set_xlabel('Total Spent ($)')

axes[1,1].hist(carts_df['days_ago'], bins=30, edgecolor='black', alpha=0.7, color='red')
axes[1,1].set_title('Cart Activity Over Time')
axes[1,1].set_xlabel('Days Ago')

plt.tight_layout()
plt.show()

print('Order count distribution:')
print(user_stats['n_carts'].value_counts().sort_index())

=== CART ANALYSIS ===
Total carts: 548
Unique users: 208
Carts per user: mean= 2.6 , max= 6
Items per user: mean= 30.3
Spend per user: mean=$ 49175.82
Order count distribution:
n_carts
1    71
2    37
3    35
4    37
5    18
6    10
Name: count, dtype: int64


## 3. Define Churn Label

In [46]:
REFERENCE_DATE = datetime.now()
INACTIVE_THRESHOLD_DAYS = 30

print('=== CHURN LABEL DEFINITION ===')
print('Reference Date:', REFERENCE_DATE)
print('Threshold:', INACTIVE_THRESHOLD_DAYS, 'days of inactivity')
print('CHURNED (1): No cart activity in last', INACTIVE_THRESHOLD_DAYS, 'days')
print('ACTIVE (0): At least one cart activity in last', INACTIVE_THRESHOLD_DAYS, 'days')

user_last_cart = carts_df.groupby('userId')['date_dt'].max()
user_days_since = ((REFERENCE_DATE - user_last_cart).dt.days).reset_index()
user_days_since.columns = ['user_id', 'days_since_last_cart']

user_days_since['churn'] = (user_days_since['days_since_last_cart'] > INACTIVE_THRESHOLD_DAYS).astype(int)

n_active = (user_days_since['churn'] == 0).sum()
n_churned = (user_days_since['churn'] == 1).sum()
total = n_active + n_churned
print('CLASS DISTRIBUTION:')
print('  ACTIVE (0):', n_active, 'users', '(' + str(round(100*n_active/total, 1)) + '%)')
print('  CHURNED (1):', n_churned, 'users', '(' + str(round(100*n_churned/total, 1)) + '%)')

=== CHURN LABEL DEFINITION ===
Reference Date: 2026-06-09 21:12:14.311034
Threshold: 30 days of inactivity
CHURNED (1): No cart activity in last 30 days
ACTIVE (0): At least one cart activity in last 30 days
CLASS DISTRIBUTION:
  ACTIVE (0): 185 users (88.9%)
  CHURNED (1): 23 users (11.1%)


## 4. Feature Engineering

**10 features** -- cart behavior and product engagement only (no date-based features).

| Domain | Features |
|--------|----------|
| Frequency | total_orders, total_items_purchased, unique_products_bought, avg_cart_size, unique_categories_bought |
| Magnitude | total_spent, avg_order_value, avg_product_rating, avg_discount_pct, avg_price_per_item |

In [47]:
engineer = EcommerceChurnFeatureEngineer(carts_df, products_df, users_df, REFERENCE_DATE)
features_df = engineer.generate_all_features()

print('Generated', len(features_df.columns), 'features for', len(features_df), 'users')
print('Feature columns:', features_df.columns.tolist())
print()
print('Feature statistics:')
print(features_df.describe().round(2))
print()
print('Feature correlations with churn:')
merged = features_df.merge(user_days_since[['user_id', 'churn']], on='user_id')
for col in features_df.columns:
    if col == 'user_id': continue
    corr = merged[col].corr(merged['churn'])
    print(f'  {col:30s} {corr:+.4f}')

Generated 14 features for 208 users
Feature columns: ['user_id', 'days_since_last_cart', 'is_active_last_7d', 'is_active_last_30d', 'total_orders', 'total_items_purchased', 'unique_products_bought', 'avg_cart_size', 'unique_categories_bought', 'total_spent', 'avg_order_value', 'avg_product_rating', 'avg_discount_pct', 'avg_price_per_item']

Feature statistics:
       user_id  days_since_last_cart  is_active_last_7d  is_active_last_30d  \
count   208.00                208.00             208.00              208.00   
mean    104.50                 12.54               0.57                0.89   
std      60.19                 16.75               0.50                0.31   
min       1.00                 -1.00               0.00                0.00   
25%      52.75                  2.00               0.00                1.00   
50%     104.50                  6.00               1.00                1.00   
75%     156.25                 17.00               1.00                1.00   
max  

## 5. Prepare Data for Modeling

In [48]:
data_df = features_df.merge(user_days_since[['user_id', 'churn']], on='user_id')

FEATURE_NAMES = [
    'total_orders',
    'total_items_purchased',
    'unique_products_bought',
    'avg_cart_size',
    'unique_categories_bought',
    'total_spent',
    'avg_order_value',
    'avg_product_rating',
    'avg_discount_pct',
    'avg_price_per_item',
]

X = data_df[FEATURE_NAMES].copy()
y = data_df['churn'].copy()

print('Dataset prepared')
print('  Samples:', len(X))
print('  Features:', len(FEATURE_NAMES))
print('  Target distribution:', dict(y.value_counts()))
print('Feature columns:', FEATURE_NAMES)

Dataset prepared
  Samples: 208
  Features: 10
  Target distribution: {0: np.int64(185), 1: np.int64(23)}
Feature columns: ['total_orders', 'total_items_purchased', 'unique_products_bought', 'avg_cart_size', 'unique_categories_bought', 'total_spent', 'avg_order_value', 'avg_product_rating', 'avg_discount_pct', 'avg_price_per_item']


## 6. Feature Selection: Method 1 -- Filter Methods

In [49]:
print('=== METHOD 1: FILTER METHODS ===')

if len(X) > 1:
    correlation_matrix = X.corr().abs()
    upper_triangle = correlation_matrix.where(
        np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
    )
    to_drop_corr = [col for col in upper_triangle.columns if any(upper_triangle[col] > 0.9)]
else:
    to_drop_corr = []
print('High correlation (> 0.9):', to_drop_corr)

variances = X.var()
to_drop_variance = [col for col in X.columns if variances[col] < 0.01]
print('Near-zero variance (< 0.01):', to_drop_variance)

features_after_filter = [col for col in X.columns if col not in (to_drop_corr + to_drop_variance)]
print('Features remaining:', len(features_after_filter))
print(features_after_filter)

if len(X) > 1:
    feature_target_corr = X[features_after_filter].corrwith(y).abs().sort_values(ascending=False)
    print()
    print('Correlation with target:')
    print(feature_target_corr)

=== METHOD 1: FILTER METHODS ===
High correlation (> 0.9): ['unique_categories_bought']
Near-zero variance (< 0.01): []
Features remaining: 9
['total_orders', 'total_items_purchased', 'unique_products_bought', 'avg_cart_size', 'total_spent', 'avg_order_value', 'avg_product_rating', 'avg_discount_pct', 'avg_price_per_item']

Correlation with target:
total_orders              0.334801
total_items_purchased     0.209365
unique_products_bought    0.166718
avg_cart_size             0.155037
total_spent               0.062033
avg_order_value           0.034189
avg_price_per_item        0.021792
avg_discount_pct          0.018489
avg_product_rating        0.003161
dtype: float64


## 7. Feature Selection: Method 2 -- RFE

In [50]:
print('=== METHOD 2: RFE ===')

if len(X) >= 2:
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled_df = pd.DataFrame(X_scaled, columns=FEATURE_NAMES)

    lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
    n_select = min(5, len(FEATURE_NAMES))
    rfe = RFE(estimator=lr, n_features_to_select=n_select, step=1)
    rfe.fit(X_scaled_df, y)

    rfe_selected = [col for col, s in zip(FEATURE_NAMES, rfe.support_) if s]
    print('RFE selected (top ' + str(n_select) + '):', rfe_selected)

    rfe_ranking = pd.DataFrame({'feature': FEATURE_NAMES, 'rank': rfe.ranking_}).sort_values('rank')
    print()
    print('RFE ranking:')
    print(rfe_ranking.to_string(index=False))
else:
    print('Not enough samples for RFE.')

=== METHOD 2: RFE ===
RFE selected (top 5): ['total_orders', 'total_items_purchased', 'unique_products_bought', 'avg_cart_size', 'unique_categories_bought']

RFE ranking:
                 feature  rank
            total_orders     1
   total_items_purchased     1
  unique_products_bought     1
           avg_cart_size     1
unique_categories_bought     1
             total_spent     2
      avg_price_per_item     3
        avg_discount_pct     4
      avg_product_rating     5
         avg_order_value     6


## 8. Feature Selection: Method 3 -- Random Forest Importance

In [51]:
print('=== METHOD 3: RANDOM FOREST ===')

if len(X) >= 2:
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled_df = pd.DataFrame(X_scaled, columns=FEATURE_NAMES)

    rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', max_depth=10)
    rf.fit(X_scaled_df, y)

    rf_importances = pd.DataFrame({'feature': FEATURE_NAMES, 'importance': rf.feature_importances_}).sort_values('importance', ascending=False)
    print('RF Feature Importance:')
    print(rf_importances.to_string(index=False))

    plt.figure(figsize=(10, 5))
    sns.barplot(data=rf_importances, x='importance', y='feature', palette='viridis')
    plt.title('Random Forest Feature Importance')
    plt.tight_layout()
    plt.show()

    n_folds = min(5, len(X))
    rf_cv_scores = cross_val_score(rf, X_scaled_df, y, cv=n_folds, scoring='f1_weighted')
    print('RF CV F1 (' + str(n_folds) + '-fold):', round(rf_cv_scores.mean(), 4), '+/-', round(rf_cv_scores.std(), 4))
else:
    print('Not enough samples for Random Forest.')

=== METHOD 3: RANDOM FOREST ===
RF Feature Importance:
                 feature  importance
            total_orders    0.236267
   total_items_purchased    0.170836
      avg_price_per_item    0.115363
      avg_product_rating    0.094636
         avg_order_value    0.088505
        avg_discount_pct    0.086307
           avg_cart_size    0.069468
             total_spent    0.067201
  unique_products_bought    0.046159
unique_categories_bought    0.025258
RF CV F1 (5-fold): 0.8709 +/- 0.0219


## 9. Feature Selection: Method 4 -- Decision Tree Importance

In [52]:
print('=== METHOD 4: DECISION TREE ===')

if len(X) >= 2:
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled_df = pd.DataFrame(X_scaled, columns=FEATURE_NAMES)

    dt = DecisionTreeClassifier(max_depth=5, random_state=42, class_weight='balanced')
    dt.fit(X_scaled_df, y)

    dt_importances = pd.DataFrame({'feature': FEATURE_NAMES, 'importance': dt.feature_importances_}).sort_values('importance', ascending=False)
    print('DT Feature Importance:')
    print(dt_importances.to_string(index=False))

    plt.figure(figsize=(10, 5))
    sns.barplot(data=dt_importances[dt_importances['importance'] > 0], x='importance', y='feature', palette='rocket')
    plt.title('Decision Tree Feature Importance')
    plt.tight_layout()
    plt.show()

    n_folds = min(5, len(X))
    dt_cv_scores = cross_val_score(dt, X_scaled_df, y, cv=n_folds, scoring='f1_weighted')
    print('DT CV F1 (' + str(n_folds) + '-fold):', round(dt_cv_scores.mean(), 4), '+/-', round(dt_cv_scores.std(), 4))
else:
    print('Not enough samples for Decision Tree.')

=== METHOD 4: DECISION TREE ===
DT Feature Importance:
                 feature   importance
            total_orders 6.358083e-01
      avg_price_per_item 1.481884e-01
        avg_discount_pct 1.024869e-01
   total_items_purchased 5.905686e-02
      avg_product_rating 5.445954e-02
           avg_cart_size 2.348894e-15
             total_spent 8.241732e-16
  unique_products_bought 0.000000e+00
unique_categories_bought 0.000000e+00
         avg_order_value 0.000000e+00
DT CV F1 (5-fold): 0.7845 +/- 0.0153


## 10. Consensus: All Methods Comparison

In [53]:
print('=== FEATURE SELECTION COMPARISON ===')

comparison = pd.DataFrame([
    {'Method': 'Full Features', 'n_features': len(FEATURE_NAMES), 'cv_f1': str(round(rf_cv_scores.mean(), 4)) if len(rf_cv_scores) > 0 else 'N/A'},
    {'Method': 'Filter', 'n_features': len(features_after_filter), 'cv_f1': 'N/A'},
    {'Method': 'RFE', 'n_features': len(rfe_selected), 'cv_f1': 'N/A'},
    {'Method': 'Random Forest', 'n_features': len(FEATURE_NAMES), 'cv_f1': str(round(rf_cv_scores.mean(), 4)) if len(rf_cv_scores) > 0 else 'N/A'},
    {'Method': 'Decision Tree', 'n_features': len(FEATURE_NAMES), 'cv_f1': str(round(dt_cv_scores.mean(), 4)) if len(dt_cv_scores) > 0 else 'N/A'},
])
print(comparison.to_string(index=False))

print()
print('Top 3 RF features:', rf_importances.head(3)['feature'].tolist())
print('Top 3 DT features:', dt_importances.head(3)['feature'].tolist())
print('RFE selected:', rfe_selected)

=== FEATURE SELECTION COMPARISON ===
       Method  n_features  cv_f1
Full Features          10 0.8709
       Filter           9    N/A
          RFE           5    N/A
Random Forest          10 0.8709
Decision Tree          10 0.7845

Top 3 RF features: ['total_orders', 'total_items_purchased', 'avg_price_per_item']
Top 3 DT features: ['total_orders', 'avg_price_per_item', 'avg_discount_pct']
RFE selected: ['total_orders', 'total_items_purchased', 'unique_products_bought', 'avg_cart_size', 'unique_categories_bought']


## 11. Final Feature Selection

In [54]:
print('=== FINAL DECISION ===')
print('Selected: All', len(FEATURE_NAMES), 'features')
print()
print('Rationale:')
print('  1. Small feature set (10) -- no need for aggressive reduction')
print('  2. All features are cart behavior / product engagement (no date reliance)')
print('  3. total_orders is the strongest predictor -- validates multi-order data')
print()
print('Final features:')
for i, feat in enumerate(FEATURE_NAMES, 1):
    imp = rf_importances[rf_importances['feature']==feat]['importance'].values[0]
    print(f'  {i:2d}. {feat:30s} importance={imp:.4f}')

=== FINAL DECISION ===
Selected: All 10 features

Rationale:
  1. Small feature set (10) -- no need for aggressive reduction
  2. All features are cart behavior / product engagement (no date reliance)
  3. total_orders is the strongest predictor -- validates multi-order data

Final features:
   1. total_orders                   importance=0.2363
   2. total_items_purchased          importance=0.1708
   3. unique_products_bought         importance=0.0462
   4. avg_cart_size                  importance=0.0695
   5. unique_categories_bought       importance=0.0253
   6. total_spent                    importance=0.0672
   7. avg_order_value                importance=0.0885
   8. avg_product_rating             importance=0.0946
   9. avg_discount_pct               importance=0.0863
  10. avg_price_per_item             importance=0.1154


## 12. Train Final Model

In [55]:
print('=== TRAINING FINAL MODEL ===')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y if y.nunique() > 1 else None
)
print('Train:', len(X_train), ', Test:', len(X_test))

final_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced')
final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)
print('Classification Report:')
print(classification_report(y_test, y_pred, zero_division=0))

if len(X) >= 4:
    n_folds = min(5, len(X))
    cv = cross_val_score(final_model, X, y, cv=n_folds, scoring='f1_weighted')
    print('CV F1:', round(cv.mean(), 4), '+/-', round(cv.std(), 4))

importances = pd.DataFrame({'feature': FEATURE_NAMES, 'importance': final_model.feature_importances_}).sort_values('importance', ascending=False)
print()
print('Final Feature Importances:')
print(importances.to_string(index=False))

=== TRAINING FINAL MODEL ===
Train: 166 , Test: 42
Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.95      0.95        37
           1       0.60      0.60      0.60         5

    accuracy                           0.90        42
   macro avg       0.77      0.77      0.77        42
weighted avg       0.90      0.90      0.90        42

CV F1: 0.8709 +/- 0.0219

Final Feature Importances:
                 feature  importance
            total_orders    0.227836
   total_items_purchased    0.179823
      avg_price_per_item    0.104979
         avg_order_value    0.101051
      avg_product_rating    0.090192
           avg_cart_size    0.081052
             total_spent    0.080971
        avg_discount_pct    0.078453
  unique_products_bought    0.034333
unique_categories_bought    0.021311


## 13. Save Model & Features

In [56]:
output_dir = '../app'
model_path = os.path.join(output_dir, 'model.pkl')
features_path = os.path.join(output_dir, 'features.json')

joblib.dump(final_model, model_path)
print('Model saved to', model_path)

meta = {
    'feature_names': FEATURE_NAMES,
    'model_type': 'RandomForestClassifier',
    'n_estimators': 100,
    'max_depth': 10,
    'churn_threshold_days': 30,
    'n_features': len(FEATURE_NAMES),
    'training_samples': len(X_train),
}
with open(features_path, 'w') as f:
    json.dump(meta, f, indent=2)
print('Features metadata saved to', features_path)
print()
print('Done! Model ready for deployment.')

Model saved to ../app\model.pkl
Features metadata saved to ../app\features.json

Done! Model ready for deployment.
